# S6 생성 파이프라인 확인

naive RAG 파이프라인(질의 → 검색 → 근거 구성 → 프롬프트 → Gemma 4 31B 생성)을 단계별로 실행해 확인한다. 답변 평가 기준은 아직 없으므로 정량 평가는 하지 않고 동작과 출력 모양을 본다(`docs/slices/06-generate.md`).

- 질의: dev 질의 중 seed로 무작위 3개(검색 결과를 보고 고르지 않음)와 회의록과 무관한 자유 질의 1개. dev 질의는 Google AI Studio 무료 쿼터로 보낼 수 있다(D-10)
- 각 호출은 LangSmith에 추적된다. 이 노트북은 기록 파일(`data/runs/generate/`)을 새로 쓰지 않는다

In [1]:
import json
from pathlib import Path

import openai
import pandas as pd
from omegaconf import OmegaConf

from rag.eval.metrics import load_gold
from rag.generate import Pipeline, recipe
from rag.index import load_cfg

In [2]:
ROOT = Path.cwd().parent
cfg = load_cfg(["index.scope=full"])
FREE_QUERY = "내일 서울의 날씨와 미세먼지 농도는 어떤가요?"

In [3]:
def context_table(contexts, gold_doc=None):
    df = pd.DataFrame(contexts)[["n", "doc_id", "chunk_id", "score", "date", "committee_name"]]
    if gold_doc is not None:
        df["정답 문서"] = df["doc_id"] == gold_doc
    return df.round({"score": 4})

## 1. 설정과 레시피

In [4]:
print(OmegaConf.to_yaml(cfg.generator))
print(OmegaConf.to_yaml(cfg.prompt))

base_url: https://generativelanguage.googleapis.com/v1beta/openai/
api_key_env: GOOGLE_API_KEY
model: models/gemma-4-31b-it
context_docs: 5
stream: false
params:
  temperature: 0.0
  top_p: 1.0
  max_tokens: 1024

version: v1
system: '당신은 국회 회의록 질의응답 도우미입니다. 아래 [근거]에 있는 내용만으로 한국어로 답하세요.

  - 답변의 각 문장 끝에 그 문장의 근거 번호를 [1]처럼 붙이세요. 여러 근거면 [1][3]처럼 붙이세요.

  - 근거에 없는 내용을 추측하거나 덧붙이지 마세요.

  - 근거로 답할 수 없으면 "제공된 회의록에서 답을 찾지 못했습니다."라고 답하세요.'
context: '[{n}] {date} {committee_name} {meeting_number} {session_number}

  {text}'
user: '[근거]

  {contexts}


  [질문]

  {query}'



## 2. 확인 질의

In [5]:
with (ROOT / cfg.paths.queries_dev).open(encoding="utf-8") as f:
    dev_qids = [json.loads(line)["qid"] for line in f]
qids = pd.Series(dev_qids).sample(3, random_state=cfg.seed).tolist()
gold = load_gold(cfg, "dev-full")
print(len(dev_qids), qids)

3016 ['048896-0013', '049273-0004', '046657-0018']


## 3. 인덱스와 클라이언트 불러오기

In [6]:
pipe = Pipeline(cfg)
print(len(pipe.chunks), pipe.chunk_emb.shape, pipe.index_meta)
recipe(cfg, pipe.index_meta, stream=False)

118039 (118039, 1024) {'name': 'kure-v1-fixed-512-64', 'n_chunks': 118039}


{'git_commit': 'e0ce84e31e40527efba234128dfe43b95727fd9e',
 'git_dirty': False,
 'embedding_model': 'nlpai-lab/KURE-v1',
 'index': 'kure-v1-fixed-512-64/full',
 'n_chunks': 118039,
 'chunk_pool': 200,
 'context_docs': 5,
 'prompt_version': 'v1',
 'prompt_sha256': 'f5876a46cec1cd73357d14be2f3a3839a6b185417da713b80bacae4f843442b0',
 'model': 'models/gemma-4-31b-it',
 'params': {'temperature': 0.0, 'top_p': 1.0, 'max_tokens': 1024},
 'stream': False}

## 4. 단계별 실행 (첫 질의)

질의 원문과 임베딩. dev 질의는 S5 캐시 임베딩을 쓴다.

In [7]:
qid = qids[0]
query, q_emb = pipe.dev_query(qid)
print(qid, query)
print(q_emb.shape, float((q_emb ** 2).sum()) ** 0.5)

048896-0013 2항을 삭제하고 1항만 남겨두자는 취지는 남용 우려가 있다면 1항에 단서조항을 추가하여 대응할 수 있을텐데, 굳이 보조 역할을 강조하는 2항이 필요한지에 대한 의문이 맞나요?
(1024,) 1.0000001192092824


검색: 상위 5개 문서마다 점수가 가장 높은 청크(D-05).

In [8]:
contexts = pipe.retrieve(q_emb, cfg.generator.context_docs)
context_table(contexts, gold.loc[qid, "doc_id"])

,n,doc_id,chunk_id,score,date,committee_name,정답 문서
0,1,048896-ebc64b2aca,74917,0.7425,2018년12월18일(화),사법개혁특별위원회-검찰·경찰개혁소위원회,True
1,2,046929-8d33debd90,46424,0.6313,2017년1월19일(목),법제사법위원회-법안심사제2소위원회,False
2,3,050775-05b4086da4,99387,0.6243,2021년2월17일(수),국토교통위원회-교통법안심사소위원회,False
3,4,046870-853f1cbb40,45573,0.6238,2016년12월29일(목),법제사법위원회-법안심사제2소위원회,False
4,5,051181-fd92809679,102297,0.6237,2021년8월17일(화),교육위원회-법안심사소위원회,False


프롬프트: 근거 번호와 회의 정보, 청크 원문. URL은 넣지 않는다.

In [9]:
messages = pipe.messages(contexts, query)
print("[system]\n" + messages[0]["content"])
print("\n[user] (앞 1,200자)\n" + messages[1]["content"][:1200])
print("\nuser 메시지 길이:", len(messages[1]["content"]), "자")

[system]
당신은 국회 회의록 질의응답 도우미입니다. 아래 [근거]에 있는 내용만으로 한국어로 답하세요.
- 답변의 각 문장 끝에 그 문장의 근거 번호를 [1]처럼 붙이세요. 여러 근거면 [1][3]처럼 붙이세요.
- 근거에 없는 내용을 추측하거나 덧붙이지 마세요.
- 근거로 답할 수 없으면 "제공된 회의록에서 답을 찾지 못했습니다."라고 답하세요.

[user] (앞 1,200자)
[근거]
[1] 2018년12월18일(화) 사법개혁특별위원회-검찰·경찰개혁소위원회 제365회 제3차
지금 함진규 위원님 안은, 조금 저는 다른 게…… 함진규 위원님은 아예 2항을 없애자는 거잖아요, 그냥 1항만 놔두자는 취지이신 것이지요? 그렇게 말씀하셨지요? 남용이 문제가 된다면, 지금 남용 때문에 그러시는 것 같은데 남용이 문제가 된다면 1항 단서조항을 달든가 하면 될 것이고, 굳이 보조하는 것을 여기다 ‘너희들은 보조만 해야 된다’ 이렇게 얘기하는 조항 자체가 나는 좀 문제가 있지 않나 이걸 얘기하는 겁니다.

[2] 2017년1월19일(목) 법제사법위원회-법안심사제2소위원회 제348회 제1차
위원장님 말씀대로 1항ㆍ2항을 그렇게 하게 되면, 개정안에 3항이 있잖아요. 3항 ‘1항 및 2항의 보험료 지원에 관한 것은 대통령령으로 정한다’ 부분이 그대로 또 남아 있지 않습니까? 그렇게 올릴 수는 있다고 보는데, 그러면 3항을 정리해야 될 것 같은데? ‘지원할 수 있다’로 해서 그렇게 합시다. 의견을 좁혀 봐요, 그러지 마시고요. 다 일방적으로 할 수는 없으니까.  2호를 살렸으면 하는 것에 대해서는 큰 이론이 없는데, 지금 절차적인 문제가 있어서 우리 전문위원은 2호만 떼서 통과시킬 수는 없다는 것이잖아요. 그러면 1호를 다 폐기해야 된다는 것이지요? 예. 그러면 방법이 없네요.   같이 반려해서 이것을 분리해서 다시 오든가 하는 방향으로 해야 되겠습니다.   그렇습니까, 전문위원? 제 의견은, 사실은 ‘지원한다’ 이런 규정이 전혀 없는 것도 아니고요. 어때요, 국방부? 

생성: Gemma 4 31B는 사고 과정을 끌 수 없어 응답 앞에 붙여 낸다. 사고 과정과 답변을 나눠 받는다.

In [10]:
result = pipe.generate(messages, stream=False)
print("[answer]\n" + result["answer"])
print("\n[thought] (앞 400자, 전체", len(result["thought"]), "자)\n" + result["thought"][:400])
print("\nusage", result["usage"])

[answer]
네, 맞습니다. 함진규 위원님의 안은 2항을 없애고 1항만 놔두자는 취지이며 [1], 남용이 문제가 된다면 1항에 단서조항을 달면 될 것이기에 굳이 '보조만 해야 된다'고 얘기하는 조항(2항)이 있는 것에 대해 문제가 있다는 의견이 제시되었습니다 [1].

[thought] (앞 400자, 전체 2596 자)
*   Question: Is it correct that the intention to delete paragraph 2 and keep only paragraph 1 is based on the idea that if there are concerns about abuse, a proviso can be added to paragraph 1, and thus there is a question about whether paragraph 2, which emphasizes the auxiliary role, is necessary?
    *   Constraint: Use only the provided [근거] (evidence).
    *   Constraint: Append evidence num

usage {'completion_tokens': 86, 'prompt_tokens': 1771, 'total_tokens': 2618, 'completion_tokens_details': None, 'prompt_tokens_details': None}


근거 목록과 원문 링크.

In [11]:
for c in contexts:
    print(f"[{c['n']}] {c['date']} {c['committee_name']}\n    {c['original']}")

[1] 2018년12월18일(화) 사법개혁특별위원회-검찰·경찰개혁소위원회
    http://likms.assembly.go.kr/record/mhs-10-040-0040.do?conferNum=048896&fileType=PDF
[2] 2017년1월19일(목) 법제사법위원회-법안심사제2소위원회
    http://likms.assembly.go.kr/record/mhs-10-040-0040.do?conferNum=046929&fileType=PDF
[3] 2021년2월17일(수) 국토교통위원회-교통법안심사소위원회
    http://likms.assembly.go.kr/record/mhs-10-040-0040.do?conferNum=050775&fileType=PDF
[4] 2016년12월29일(목) 법제사법위원회-법안심사제2소위원회
    http://likms.assembly.go.kr/record/mhs-10-040-0040.do?conferNum=046870&fileType=PDF
[5] 2021년8월17일(화) 교육위원회-법안심사소위원회
    http://likms.assembly.go.kr/record/mhs-10-040-0040.do?conferNum=051181&fileType=PDF


## 5. 나머지 질의 (한 번에 실행)

API 오류가 나면 그 질의는 오류를 기록하고 다음 질의로 넘어간다. 질의 `046657-0018`은 이 노트북의 이전 실행과 CLI에서 반복해 500 `INTERNAL`이 났다(프롬프트 캐시가 적중했던 CLI 1회만 성공).

In [12]:
records, failed = [], {}
for qid in qids[1:]:
    query, q_emb = pipe.dev_query(qid)
    try:
        rec = pipe.run(query, q_emb, qid=qid)
    except openai.APIStatusError as e:
        failed[qid] = f"{e.status_code} {e.message[:80]}"
        print(f"[{qid}] {query}\n→ 오류 {failed[qid]}\n")
        continue
    records.append(rec)
    print(f"[{qid}] {query}\n→ {rec['answer']}\n")

[049273-0004] 의견 결정이 길어지는 경우, 무응답이 아닌 간주 규정을 어떻게 적용할지에 대한 명확한 기준이 필요하다고 생각하시나요?
→ 의견을 결정하지 못해 보류했을 때 간주 규정이 적용되는지에 대하여, 명확성을 기하기 위해 이 부분을 짚고 넘어가는 것이 어떻겠냐는 의견이 제시되었습니다 [1]. 또한 보류는 법상 인정되는 행위가 아닌 것으로 보이며, 그런 경우에는 연장 규정을 적용하는 것이라고 언급되었습니다 [1].



[046657-0018] 현재 사건에서 한 쪽은 병사, 한 쪽은 외인사로 주장하고 있는데, 이로 인해 혼란이 생기고 있습니다. 외인사 주장이라면 부검이 필요할 것으로 생각됩니다. 지검장님, 이에 대한 의견을 듣고 계시나요?
→ 지검장님은 위원님의 말씀에 전적으로 공감하며, 그렇기 때문에 영장을 두 번이나 청구했다고 답변하였습니다 [2]. 또한 지금까지 사회적인 이슈가 되었던 사건들에 대해 부검이 왜곡되거나 사인을 조작한 바가 없으며, 시위 현장에서 명을 달리하신 분들에 대해서는 예외 없이 다 부검을 했다고 설명하였습니다 [2].



In [13]:
for rec in records:
    display(context_table(rec["contexts"], gold.loc[rec["qid"], "doc_id"]))

,n,doc_id,chunk_id,score,date,committee_name,정답 문서
0,1,049273-95fd0ca8ba,79086,0.7080,2019년7월22일(월),산업통상자원중소벤처기업위원회-산업통상자원특허소위원회,True
1,2,047025-7745a5efb8,47251,0.6652,2017년2월20일(월),교육문화체육관광위원회-문화체육관광법안심사소위원회,False
2,3,047019-930f274f76,47156,0.6223,2017년2월20일(월),산업통상자원위원회-법률안소위원회,False
3,4,049273-c10466bf21,79089,0.6212,2019년7월22일(월),산업통상자원중소벤처기업위원회-산업통상자원특허소위원회,False
4,5,047608-6bd8a2f807,54410,0.6202,2017년11월22일(수),헌법재판소장(이진성)임명동의에관한인사청문특별위원회,False


,n,doc_id,chunk_id,score,date,committee_name,정답 문서
0,1,046615-69ab5e3bc6,41191,0.7125,2016년10월11일(화),교육문화체육관광위원회,False
1,2,046657-37f4ccc6b4,41878,0.6738,2016년10월4일(화),법제사법위원회,True
2,3,046685-303a2eccb2,42702,0.6514,2016년10월5일(수),법제사법위원회,False
3,4,046662-8b38971202,42097,0.6455,2016년10월11일(화),법제사법위원회,False
4,5,046473-b723ca92e9,37000,0.6444,2016년10월6일(목),안전행정위원회,False


## 6. 회의록과 무관한 자유 질의

데이터셋에 없는 질의라 KURE로 임베딩한다. 근거로 답할 수 없으면 그렇다고 답하는지 본다.

In [14]:
try:
    free = pipe.run(FREE_QUERY, pipe.embed(FREE_QUERY))
    print(FREE_QUERY, "\n→", free["answer"])
except openai.APIStatusError as e:
    free, failed["자유 질의"] = None, f"{e.status_code} {e.message[:80]}"
    print(FREE_QUERY, "\n→ 오류", failed["자유 질의"])
context_table(free["contexts"]) if free else None

C:\workspace\meet-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:  60%|██████    | 236/391 [00:00<00:00, 2347.75it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2606.62it/s]

내일 서울의 날씨와 미세먼지 농도는 어떤가요? 
→ 제공된 회의록에서 답을 찾지 못했습니다.


,n,doc_id,chunk_id,score,date,committee_name
0,1,049839-0807e703d7,90063,0.5655,2019년10월7일(월),환경노동위원회
1,2,048118-683fb21ff5,62985,0.5644,2018년4월9일(월),미세먼지대책특별위원회
2,3,047923-fe68925e0b,61091,0.5639,2018년1월17일(수),미세먼지대책특별위원회
3,4,047491-46ad314a1a,51614,0.5630,2017년10월13일(금),환경노동위원회
4,5,048988-4f90876e4c,76133,0.5593,2019년3월11일(월),환경노동위원회-환경소위원회


## 7. 정리

In [15]:
rows = []
for name, rec in [(qids[0], {"contexts": contexts, **result}), *[(r["qid"], r) for r in records],
                  *([("자유 질의", free)] if free else [])]:
    docs = [c["doc_id"] for c in rec["contexts"]]
    g = gold.loc[name, "doc_id"] if name in gold.index else None
    rows.append({"질의": name, "정답 문서 위치": docs.index(g) + 1 if g in docs else None,
                 "인용 번호 있음": "[" in rec["answer"], "답변 길이": len(rec["answer"]),
                 "사고 과정 길이": len(rec["thought"]),
                 "전체 토큰": rec["usage"]["total_tokens"]})
print("API 오류:", failed)
pd.DataFrame(rows)

API 오류: {}


,질의,정답 문서 위치,인용 번호 있음,답변 길이,사고 과정 길이,전체 토큰
0,048896-0013,1.0,True,142,2596,2618
1,049273-0004,1.0,True,158,2828,3225
2,046657-0018,2.0,True,171,2713,2924
3,자유 질의,NaN,False,22,1688,3590


- 정답 문서 위치는 근거 5개 안에서 정답 문서가 몇 번째인지다. 답변 평가 기준을 정하기 전이라 판단에 쓰지 않고 관찰로만 남긴다
- 사고 과정 토큰은 `usage`의 전체 토큰에 포함된다(입력·출력 토큰 합보다 큰 부분). 무료 등급 TPM 16K에 함께 들어간다

LangSmith로 보낸 추적이 커널 종료 전에 모두 전송되게 기다린다.

In [16]:
from langsmith.run_trees import get_cached_client

get_cached_client().flush()